# Synchronous document processing with standard output
In the previous notebooks we used `InvokeDataAutomationAsync`, which reads the input from S3, writes results to an S3 output prefix and requires polling for job completion. Bedrock Data Automation (BDA) also offers a synchronous API, `InvokeDataAutomation`, that processes the input and returns structured results directly in the API response. Both APIs support standard output and custom output using blueprints, so you can pick the one that fits your application's latency requirements.

## When to use synchronous processing
Synchronous processing is designed for low-latency, request/response workloads where a user or an upstream service is waiting for the result. Typical scenarios include:
* **Interactive applications** - extract insights while the user waits. Examples: identifying products from a photo a customer uploads to an e-commerce app, recognizing landmarks in a travel app, or reading the details from an ID card or receipt during an onboarding or expense flow.
* **Inline content moderation** - analyze user-uploaded images before they are published, for example on a social media platform.
* **Real-time validation and routing** - check a single-page document (such as a form or bank statement) as it arrives and immediately decide the next step, for example accepting it, asking for a re-upload, or routing it to a reviewer.
* **Simpler architectures** - there is no job to poll, no completion events or callbacks to handle, and no output to read back from S3. This makes sync mode a good fit for APIs, AWS Lambda functions and agent tools that need an answer in a single call.

Some useful details about the synchronous API:
* The input can come from an S3 URI or be passed directly as bytes in the request.
* Results are returned inline in the response by default. You can optionally provide an S3 output location to store them instead.
* The response contains a list of `outputSegments`, one for each logical sub-document, each carrying its standard output (and custom output, if blueprints are used).
* The project must be created with `projectType='SYNC'`.

For large multi-page files, batch or bulk processing, and audio and video content, use the asynchronous `InvokeDataAutomationAsync` API covered in the earlier notebooks.

Sources: [InvokeDataAutomation API reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomation.html), [Amazon Bedrock Data Automation now supports synchronous image processing](https://aws.amazon.com/about-aws/whats-new/2025/11/bedrock-data-automation-synchronous-image-processing/), [Bedrock Data Automation User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/bda.html)

## What this notebook covers
* Create (or update) a data automation project with `projectType='SYNC'` and a standard output configuration
* Invoke BDA synchronously on a sample bank statement image
* Explore the standard output returned in the response

## Prerequisites

In [ ]:
%pip install --no-warn-conflicts "boto3>=1.42.35" itables==2.2.4 PyPDF2==3.0.1 --upgrade -qq

## Setup

Before we get to the part where we invoke BDA with our sample artifacts, let's setup some parameters and configuration that will be used throughout this notebook.

In [ ]:
import boto3
import json
from IPython.display import JSON, display, IFrame
import sagemaker
import pandas as pd
from utils import helper_functions
from utils import display_functions
import os

session = sagemaker.Session()
default_bucket = session.default_bucket()
current_region = boto3.session.Session().region_name
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']

# Initialize Bedrock Data Automation clients
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
s3_client = boto3.client('s3')

bda_s3_input_location = f's3://{default_bucket}/bda/input'

## Prepare sample document
For this lab, we use a sample `Bank Statement`. The statement contains account holder information, bank account details and a balance summary along with a table of transactions. We upload the document to S3 and process it synchronously using standard output.

In [ ]:
local_file_path = 'data/documents/BankStatement.jpg'
local_file_name = os.path.basename(local_file_path)

document_s3_uri = f'{bda_s3_input_location}/{local_file_name}'
target_s3_bucket, target_s3_key = helper_functions.get_bucket_and_key(document_s3_uri)
s3_client.upload_file(local_file_path, target_s3_bucket, target_s3_key)

print(f"Uploaded file to S3: {target_s3_key}")
print(f"document_s3_uri: {document_s3_uri}")

### View Sample Document

In [ ]:
IFrame(local_file_path, width=1000, height=400)

### Create a SYNC data project
Next we create the data project. If a project with the given name already exists, we update it with the given configuration; otherwise we create a new one.
A few points to note -
* The project is created with `projectType='SYNC'`, which is required to use the synchronous `InvokeDataAutomation` API
* We enable all granularity types (`DOCUMENT`, `PAGE`, `ELEMENT`, `LINE`, `WORD`), bounding boxes, generative fields and markdown text output in the standard output configuration
* No blueprints are attached, so the response contains standard output only

In [ ]:
bda_project_name = 'bda-workshop-standard-output-sync'
bda_project_stage = 'LIVE'
standard_output_configuration = {
    'document': {
        'extraction': {
            'granularity': {
                'types': [
                    'DOCUMENT', 'PAGE', 'ELEMENT', 'LINE', 'WORD'
                ]
            },
            'boundingBox': {
                'state': 'ENABLED'
            }
        },
        'generativeField': {
            'state': 'ENABLED'
        },
        'outputFormat': {
            'textFormat': {
                'types': [
                    'MARKDOWN'
                ]
            },
            'additionalFileFormat': {
                'state': 'DISABLED'
            }
        }
    }
}
override_configuration = {}
custom_output_configuration = {}

In [ ]:
list_project_response = bda_client.list_data_automation_projects(
    projectStageFilter=bda_project_stage)
project = next((project for project in list_project_response['projects']
               if project['projectName'] == bda_project_name), None)
if not project:
    response = bda_client.create_data_automation_project(
        projectName=bda_project_name,
        projectType='SYNC',
        projectDescription='Synchronous document processing with standard output',
        projectStage=bda_project_stage,
        standardOutputConfiguration=standard_output_configuration,
        customOutputConfiguration=custom_output_configuration,
        overrideConfiguration=override_configuration
    )
else:
    response = bda_client.update_data_automation_project(
        projectArn=project['projectArn'],
        standardOutputConfiguration=standard_output_configuration,
        customOutputConfiguration=custom_output_configuration,
        overrideConfiguration=override_configuration
    )
project_arn = response['projectArn']

### Wait for create/update data project operation completion

In [ ]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_client.get_data_automation_project,
            status_kwargs={'projectArn': project_arn},
            completion_states=['COMPLETED'],
            error_states=['FAILED'],
            status_path_in_response='project.status',
            max_iterations=15,
            delay=30
)

### Invoke Data Automation Sync
With the data project configured, we can now invoke data automation for our sample document. Unlike `InvokeDataAutomationAsync`, the synchronous `InvokeDataAutomation` call blocks until processing finishes and returns the results inline in the response, so there is no job to poll and no output to read back from S3.

In [ ]:
print(f"Invoking bda - input: {document_s3_uri}")
response = bda_runtime_client.invoke_data_automation(
    inputConfiguration={
        's3Uri': document_s3_uri
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': bda_project_stage
    },
    dataAutomationProfileArn=f'arn:aws:bedrock:{current_region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

### View Job Metadata
The response contains a list of `outputSegments`. Each segment carries its standard output as a JSON string. Let's parse the standard output of the first segment and look at its metadata.

In [ ]:
standard_output_s = response["outputSegments"][0]['standardOutput']
standard_output = json.loads(standard_output_s)
job_metadata = standard_output['metadata']

job_metadata_table = pd.DataFrame([job_metadata]).fillna('')
job_metadata_table.index.name='Segment Index'
job_metadata_json = JSON(job_metadata, root="job_metadata", expanded=True)
# Display the widget
display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)], 
    ["Table View", "Raw JSON"])

## Explore Document Insights using Standard output

In [ ]:
JSON(standard_output)